# Modellazione di curve di domanda retail non lineari con PROC GAM

## Sintesi

Questo notebook utilizza **PROC GAM** per modellare le vendite settimanali di unità di generi alimentari come funzione liscia e non lineare del prezzo di scaffale, della temperatura del negozio (proxy di stagionalità) e della spesa promozionale, con un effetto parametrico del flag di promozione. I modelli additivi generalizzati permettono a un category manager di recuperare le vere forme curve dell'elasticità di prezzo e della domanda stagionale che una regressione lineare non riuscirebbe a cogliere, supportando decisioni di pricing e promozione più mirate.

## Fonti dei dati

| Dataset | Righe | Grana | Variabili chiave | Descrizione |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | settimana | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Serie settimanale sintetica di dati di vendita al punto cassa per un negozio alimentare di punta lungo 100 settimane consecutive (quasi due cicli stagionali). Generata inline con `call streaminit(20250531)` e `rand()`. Le vendite settimanali di unità seguono un processo di domanda di Poisson la cui media è determinata da una curva di risposta al prezzo esponenziale, un effetto quadratico di temperatura/stagionalità con picco vicino a 72F, e un incremento concavo (radice quadrata) della spesa promozionale, più un flag discreto di settimana promozionale. |

# Modellazione di curve di domanda retail non lineari con PROC GAM

La domanda al dettaglio raramente risponde al prezzo, al meteo o alla spesa promozionale in modo lineare. Un piccolo taglio di prezzo su un prodotto di base può spostare a malapena il volume, mentre superare una soglia psicologica di prezzo può innescare un salto brusco; la domanda guidata dal meteo raggiunge il picco in un intervallo confortevole intermedio e diminuisce ai due estremi; e l'incremento promozionale mostra rendimenti decrescenti all'aumentare della spesa.

**PROC GAM** (modelli additivi generalizzati) adatta a ogni driver un proprio termine spline liscio, cosicché sono i dati — non un'assunzione lineare rigida — a determinare la forma di ogni curva di domanda. Qui modelliamo le vendite settimanali di unità per un singolo negozio alimentare di punta lungo 100 settimane consecutive, combinando un flag di promozione parametrico con spline di lisciamento su prezzo, temperatura e spesa promozionale sotto una risposta di Poisson.

## Passo 1 — Generazione di una serie sintetica di vendite settimanali

Simuliamo 100 settimane consecutive (quasi due cicli stagionali) di dati al punto cassa per un negozio di punta. Il processo di generazione dei dati è deliberatamente non lineare, cosicché possiamo verificare che GAM recuperi forme realistiche:

- Il **Prezzo** guida il volume attraverso una curva di risposta esponenziale (`exp(-1.1 * Price)`), per cui la domanda cresce ripidamente al calare del prezzo.
- La **Temperatura** funge da proxy di stagionalità con un picco quadratico vicino a 72F, modellando l'affluenza guidata dal comfort.
- La **Spesa promozionale** fornisce un incremento concavo a radice quadrata (rendimenti decrescenti).
- Un flag discreto di **Promozione** aggiunge un effetto a gradino parametrico nelle settimane promozionate.

Le `Units` (unità) settimanali sono estratte da una distribuzione di Poisson, coerente con la natura di conteggio delle vendite di unità.

In [1]:
DATI store_sales;
   CHIAMARE streaminit(20250531);
   LUNGHEZZA Promotion $3;
   FARE Week = 1 FINO_A 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      SE_COND rand("uniform") < 0.28 ALLORA FARE;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      FINE;
      ALTRIMENTI FARE;
         Promotion  = "No";
         PromoSpend = 0;
      FINE;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      USCITA;
   FINE;
ESEGUIRE;


NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Passo 2 — Profilazione dei dati simulati

Una rapida `PROC MEANS` conferma che le variabili coprono intervalli sensati per il retail prima della modellazione: i conteggi di unità sono interi non negativi, il prezzo si concentra attorno a pochi dollari, la temperatura percorre un intero ciclo stagionale e la spesa promozionale è zero nelle settimane non promozionate.

In [2]:
PROCEDURA MEDIE DATI=store_sales n mean MIN MAX maxdec=2;
   VARIABILE Units Price Temperature PromoSpend;
   ETICHETTA Units="Unità" Price="Prezzo" Temperature="Temperatura" PromoSpend="Spesa promozionale";
ESEGUIRE;

                                                  The MEANS Procedure

 Variable     Label                      N           Mean     Minimum     Maximum
 --------------------------------------------------------------------------------
 Units        Unità                    100           7.03        0.00      103.00
 Price        Prezzo                   100           3.15        2.81        3.48
 Temperature  Temperatura              100          55.50       22.72       83.49
 PromoSpend   Spesa promozionale       100         128.76        0.00      779.00
 --------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Passo 3 — Adattamento del modello additivo di domanda completo

Il modello principale combina:

- `param(Promotion)` — un effetto parametrico (lineare) per l'indicatore di settimana promozionata, dichiarato nell'istruzione `CLASS`.
- `spline(Price, df=4)` — una spline di lisciamento cubica che cattura la risposta curva al prezzo.
- `spline(Temperature, df=4)` — una curva liscia di stagionalità.
- `spline(PromoSpend, df=3)` — l'incremento promozionale a rendimenti decrescenti.

Poiché le vendite di unità sono conteggi, specifichiamo `dist=poisson` (link log). `method=gcv` lascia che la convalida incrociata generalizzata guidi la levigatezza di ogni componente senza un override esplicito dei gradi di libertà. L'istruzione `OUTPUT` scrive le previsioni e i residui per osservazione in `gam_fit`.

La procedura riporta una **Devianza di 43.59** contro una **Devianza nulla di 2041.12** — i quattro driver lisci più parametrico spiegano quasi tutta la variazione che un modello con la sola costante lascerebbe inspiegata — e un **AIC di 254.61**. La stima parametrica `PROMOTIONYES` è positiva (+0.41 sulla scala logaritmica), confermando l'incremento di volume promozionale, e la spline del prezzo presenta una tendenza lineare fortemente negativa (-1.71), la firma tipica di una domanda decrescente al crescere del prezzo.

In [3]:
PROCEDURA gam DATI=store_sales PLOTS=components(CLM commonaxes);
   CLASSE Promotion;
   MODELLO Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   USCITA out=gam_fit predicted residual;
   ETICHETTA Units="Unità" Promotion="Promozione" Price="Prezzo" Temperature="Temperatura" PromoSpend="Spesa promozionale";
ESEGUIRE;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Unità
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Prezzo)                   4.0000         4.


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Passo 4 — Un modello mirato su prezzo e stagionalità

Per una revisione di pricing più snella, riadattiamo il modello con solo i due driver lisci più rilevanti per le decisioni — prezzo e temperatura — dando al prezzo flessibilità extra (`df=5`) per risolvere eventuali flessioni vicino a una soglia psicologica di prezzo. Il flag di promozione è mantenuto come controllo parametrico.

Eliminare la spesa promozionale porta la **Devianza a 62.80** e l'**AIC a 269.82**, entrambi più alti dei valori 43.59 e 254.61 del modello completo. Il termine parametrico `PROMOTIONYES` assorbe qui anche più segnale promozionale (+1.73 contro +0.41), poiché la spline della spesa non è più presente a portarne parte. La spline del prezzo mantiene la sua tendenza negativa (-1.51), per cui la storia di fondo della domanda resta stabile tra le specificazioni.

In [4]:
PROCEDURA gam DATI=store_sales PLOTS=components(CLM);
   CLASSE Promotion;
   MODELLO Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   ETICHETTA Units="Unità" Promotion="Promozione" Price="Prezzo" Temperature="Temperatura";
ESEGUIRE;


                                                   The GAM Procedure                                                    

Model Information
Response Variable     Unità
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Prezzo)                   5.0000         5.0000
Spline(Temperatura)              4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Interpretazione dei risultati

La tabella **Regression Model Analysis** riporta la tendenza lineare all'interno di ogni componente più l'effetto parametrico di promozione. Il coefficiente positivo `PROMOTIONYES` (+0.41 nel modello completo, +1.73 in quello più snello) conferma l'atteso incremento di volume promozionale, mentre la tendenza lineare negativa sulla spline del prezzo (-1.71 e -1.51) riflette la classica domanda decrescente al crescere del prezzo. Il piccolo termine lineare positivo della spline di temperatura (+0.03) è atteso: la sua vera storia è la curvatura attorno al picco di comfort di 72F, che un singolo coefficiente lineare non può riassumere.

La tabella **Smoothing Model Analysis** riporta i gradi di libertà consumati da ogni spline. Prezzo e temperatura assorbono ciascuno 4 DF effettivi (5 per il prezzo nel modello più snello) e la spesa promozionale 3 — ciascuno ben al di sopra del singolo DF che userebbe una retta, ed è esattamente per questo che una regressione lineare non coglierebbe queste relazioni curve.

Le **Fit Statistics** permettono di confrontare direttamente le due specificazioni. Il modello completo a quattro driver riporta una Devianza di 43.59 e un AIC di 254.61 contro i 62.80 e 269.82 del modello più snello; entrambi i criteri favoriscono il modello completo, mostrando che la spesa promozionale e il flag di promozione aggiungono potere esplicativo oltre a prezzo e temperatura da soli. Rispetto alla Devianza nulla di 2041.12, entrambi i modelli catturano la stragrande maggioranza della variazione della domanda.

Lette insieme, queste tabelle offrono a un category manager una storia di domanda quantificata e guidata dai dati: una risposta di prezzo ripida che informa la profondità dei ribassi, una finestra stagionale di temperatura e un effetto di spesa promozionale a rendimenti decrescenti — una guida molto più precisa di una singola stima lineare di elasticità. (PROC GAM accetta anche `plots=components` per rendere le curve di previsione parziale per ogni termine liscio; le tabelle numeriche di componenti sopra sono la fonte da cui derivano quelle curve.)